# ResLit - PubTator3 Bulk Download + PMID Filtering Pipeline

**FTP:** ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/

## Files (~3.3GB total)
- `mutation2pubtator3.gz` - 110M (variants, SNPs) - most specific
- `gene2pubtator3.gz` - 713M (gene/protein mentions)
- `species2pubtator3.gz` - 459M (organism mentions) - metadata only
- `chemical2pubtator3.gz` - 1.7G (drug/compound mentions)
- `relation2pubtator3.gz` - 276M (entity pair relations) - strongest signal

## Filter Logic
**KEEP:** (has_gene OR has_mutation) AND has_chemical

**Tiers:**
- **Tier 1:** (gene OR mutation) AND chemical → ~400-600k (safety net)
- **Tier 2:** Tier 1 AND has_mutation → ~150-250k (LLM screener input)
- **Tier 3:** Tier 2 AND has_hv_relation → ~50-100k (priority extraction)
- **Not in PubTator:** Secondary batch, process last

In [ ]:
!pip install duckdb pandas tqdm

In [ ]:
import gzip
import logging
import subprocess
from pathlib import Path

import duckdb
import pandas as pd
from tqdm import tqdm

print("✓ All imports loaded")

✓ All imports loaded


In [ ]:
# ─── LOGGING ───
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("pubtator_filter.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─── CONFIG ───
FTP_BASE     = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3"
DOWNLOAD_DIR = Path("./pubtator3_data")
DB_PATH      = Path("./pubtator3_index.duckdb")
RESULTS_DIR  = Path("./results")

# Files to download — order matters (smallest/most specific first)
FILES = {
    "mutation": "mutation2pubtator3.gz",   # 110M
    "gene":     "gene2pubtator3.gz",       # 713M
    "species":  "species2pubtator3.gz",    # 459M
    "chemical": "chemical2pubtator3.gz",   # 1.7G
    "relation": "relation2pubtator3.gz",   # 276M
}

# Relations that strongly suggest a resistance mechanism paper
HIGH_VALUE_RELATION_PAIRS = {
    frozenset(["Gene",    "Chemical"]),
    frozenset(["Variant", "Chemical"]),
    frozenset(["Gene",    "Disease"]),
    frozenset(["Variant", "Disease"]),
}

print("✓ Configuration loaded")

✓ Configuration loaded


## Step 1: Download PubTator3 Files

In [ ]:
def download_files(download_dir: Path = DOWNLOAD_DIR):
    """
    Download PubTator3 annotation files with resume support.
    Uses wget if available, falls back to curl.
    Total: ~3.3GB
    """
    download_dir.mkdir(parents=True, exist_ok=True)

    for entity_type, filename in FILES.items():
        dest = download_dir / filename

        if dest.exists():
            log.info(f"  Already exists, skipping: {filename}")
            continue

        url = f"{FTP_BASE}/{filename}"
        log.info(f"  Downloading {filename}...")

        wget_available = subprocess.run(
            ["which", "wget"], capture_output=True
        ).returncode == 0

        if wget_available:
            cmd = ["wget", "-c", "-q", "--show-progress", "-O", str(dest), url]
        else:
            cmd = ["curl", "-C", "-", "-L", "--progress-bar", "-o", str(dest), url]

        result = subprocess.run(cmd)
        if result.returncode != 0:
            log.error(f"  Failed to download {filename}")
            raise RuntimeError(f"Download failed for {filename}")

        size_gb = dest.stat().st_size / 1e9
        log.info(f"  Done: {filename} ({size_gb:.2f} GB)")

print("✓ download_files() function defined")

✓ download_files() function defined


## Step 2: Build DuckDB Index

In [ ]:
def _stream_entity_file(gz_path: Path, entity_type: str, batch_size: int = 500_000):
    """
    Stream a gzipped PubTator3 entity file line by line.
    Yields batches of (pmid, entity_type) rows.

    Format: PMID  ConceptType  Offset  Length  Text  Identifier
    """
    batch = []
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            pmid = parts[0].strip()
            if pmid.isdigit():
                batch.append((pmid, entity_type))
                if len(batch) >= batch_size:
                    yield batch
                    batch = []
    if batch:
        yield batch

print("✓ _stream_entity_file() function defined")

✓ _stream_entity_file() function defined


In [ ]:
def _stream_relation_file(gz_path: Path, batch_size: int = 500_000):
    """
    Stream relation file, flagging high-value entity pairs.
    Format: PMID  relation_type  e1_id  e1_type  e2_id  e2_type
    """
    batch = []
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 6:
                continue
            pmid    = parts[0].strip()
            e1_type = parts[3].strip()
            e2_type = parts[5].strip()
            if not pmid.isdigit():
                continue
            is_hv = frozenset([e1_type, e2_type]) in HIGH_VALUE_RELATION_PAIRS
            batch.append((pmid, is_hv))
            if len(batch) >= batch_size:
                yield batch
                batch = []
    if batch:
        yield batch

print("✓ _stream_relation_file() function defined")

✓ _stream_relation_file() function defined


In [ ]:
def build_index(download_dir: Path = DOWNLOAD_DIR, db_path: Path = DB_PATH):
    """
    Parse all PubTator3 files and build a DuckDB index:
        pmid_entities(pmid, has_gene, has_chemical, has_species,
                      has_mutation, has_relation, has_hv_relation)
    One row per PMID. Fast boolean lookup at filter time.
    """
    con = duckdb.connect(str(db_path))

    # Raw staging tables
    con.execute("""
        CREATE TABLE IF NOT EXISTS raw_annotations (
            pmid        TEXT,
            entity_type TEXT
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS raw_relations (
            pmid          TEXT,
            is_high_value BOOLEAN
        )
    """)

    # ── Entity files ──
    entity_files = [
        ("mutation", FILES["mutation"]),
        ("gene",     FILES["gene"]),
        ("species",  FILES["species"]),
        ("chemical", FILES["chemical"]),
    ]

    for entity_type, filename in entity_files:
        gz_path = download_dir / filename
        if not gz_path.exists():
            log.warning(f"  File not found, skipping: {filename}")
            continue

        log.info(f"  Parsing {filename}...")
        total = 0
        for batch in tqdm(_stream_entity_file(gz_path, entity_type),
                          desc=f"    {entity_type}", unit="batch"):
            df = pd.DataFrame(batch, columns=["pmid", "entity_type"])
            con.execute("INSERT INTO raw_annotations SELECT * FROM df")
            total += len(batch)
        log.info(f"    → {total:,} annotations loaded")

    # ── Relation file ──
    relation_path = download_dir / FILES["relation"]
    if relation_path.exists():
        log.info(f"  Parsing {FILES['relation']}...")
        total = 0
        for batch in tqdm(_stream_relation_file(relation_path),
                          desc="    relations", unit="batch"):
            df = pd.DataFrame(batch, columns=["pmid", "is_high_value"])
            con.execute("INSERT INTO raw_relations SELECT * FROM df")
            total += len(batch)
        log.info(f"    → {total:,} relations loaded")

    # ── Pivot to one row per PMID ──
    log.info("  Building PMID summary table...")
    con.execute("""
        CREATE OR REPLACE TABLE pmid_entities AS
        SELECT
            pmid,
            MAX(CASE WHEN entity_type = 'gene'     THEN TRUE ELSE FALSE END) AS has_gene,
            MAX(CASE WHEN entity_type = 'chemical' THEN TRUE ELSE FALSE END) AS has_chemical,
            MAX(CASE WHEN entity_type = 'species'  THEN TRUE ELSE FALSE END) AS has_species,
            MAX(CASE WHEN entity_type = 'mutation' THEN TRUE ELSE FALSE END) AS has_mutation,
            FALSE AS has_relation,
            FALSE AS has_hv_relation
        FROM raw_annotations
        GROUP BY pmid
    """)

    # ── Add relation flags ──
    con.execute("""
        UPDATE pmid_entities
        SET has_relation = TRUE
        WHERE pmid IN (SELECT DISTINCT pmid FROM raw_relations)
    """)
    con.execute("""
        UPDATE pmid_entities
        SET has_hv_relation = TRUE
        WHERE pmid IN (
            SELECT DISTINCT pmid FROM raw_relations WHERE is_high_value = TRUE
        )
    """)

    # ── Cleanup + index ──
    con.execute("DROP TABLE raw_annotations")
    con.execute("DROP TABLE raw_relations")
    con.execute("CREATE INDEX IF NOT EXISTS idx_pmid ON pmid_entities(pmid)")
    con.execute("ANALYZE")

    # ── Stats ──
    def count(where=""):
        q = f"SELECT COUNT(*) FROM pmid_entities{(' WHERE ' + where) if where else ''}"
        return con.execute(q).fetchone()[0]

    log.info(f"\n  Index complete:")
    log.info(f"    Total PMIDs indexed:    {count():>10,}")
    log.info(f"    has_gene:               {count('has_gene'):>10,}")
    log.info(f"    has_chemical:           {count('has_chemical'):>10,}")
    log.info(f"    has_species:            {count('has_species'):>10,}")
    log.info(f"    has_mutation:           {count('has_mutation'):>10,}")
    log.info(f"    has_relation:           {count('has_relation'):>10,}")
    log.info(f"    has_hv_relation:        {count('has_hv_relation'):>10,}")
    log.info(f"  DuckDB index saved → {db_path}")

    con.close()

print("✓ build_index() function defined")

✓ build_index() function defined


## Execute Pipeline

In [ ]:
# Configuration for this run
seed_pmid_file = "/home/argis/Desktop/austin/pubmed_amr/amrprofiler_database/amr_genes_pmids_amrprofiler_uniq.txt"  # Change to your file
#seed_pmid_file = "/home/argis/Desktop/austin/pubmed_amr/amr_pmids_genetics2.txt"  # Change to your file
skip_download = False
skip_index = False

In [ ]:
# Step 1: Download
print("\n" + "="*60)
print("STEP 1: Download PubTator3 files (~3.3GB)")
print("="*60)

if skip_download:
    print("  Skipped (--skip-download)")
else:
    download_files(DOWNLOAD_DIR)

2026-03-12 07:35:40,188 [INFO]   Downloading mutation2pubtator3.gz...



STEP 1: Download PubTator3 files (~3.3GB)



     0K .......... .......... .......... .......... ..........  0%  564K 3m19s
    50K .......... .......... .......... .......... ..........  0% 1,10M 2m29s
   100K .......... .......... .......... .......... ..........  0% 62,4M 1m40s
   150K .......... .......... .......... .......... ..........  0%  158M 75s
   200K .......... .......... .......... .......... ..........  0% 1,11M 80s
   250K .......... .......... .......... .......... ..........  0%  125M 67s
   300K .......... .......... .......... .......... ..........  0%  192M 57s
   350K .......... .......... .......... .......... ..........  0%  171M 50s
   400K .......... .......... .......... .......... ..........  0% 1,13M 55s
   450K .......... .......... .......... .......... ..........  0%  114M 50s
   500K .......... .......... .......... .......... ..........  0%  184M 45s
   550K .......... .......... .......... .......... ..........  0%  115M 42s
   600K .......... .......... .......... .......... ..........  0%  1

In [ ]:
# Step 2: Build Index
print("\n" + "="*60)
print("STEP 2: Build DuckDB index")
print("="*60)

if skip_index:
    print("  Skipped (--skip-index)")
else:
    build_index(DOWNLOAD_DIR, DB_PATH)

2026-03-12 07:37:04,742 [INFO]   Parsing mutation2pubtator3.gz...



STEP 2: Build DuckDB index


    mutation: 0batch [00:00, ?batch/s]

    mutation: 17batch [00:19,  1.13s/batch]
2026-03-12 07:37:23,977 [INFO]     → 8,071,156 annotations loaded
2026-03-12 07:37:23,978 [INFO]   Parsing gene2pubtator3.gz...
    gene: 148batch [02:40,  1.08s/batch]
2026-03-12 07:40:04,484 [INFO]     → 73,646,988 annotations loaded
2026-03-12 07:40:04,485 [INFO]   Parsing species2pubtator3.gz...
    species: 97batch [01:52,  1.16s/batch]
2026-03-12 07:41:56,788 [INFO]     → 48,083,952 annotations loaded
2026-03-12 07:41:56,789 [INFO]   Parsing chemical2pubtator3.gz...
    chemical: 291batch [05:39,  1.17s/batch]
2026-03-12 07:47:36,055 [INFO]     → 145,064,127 annotations loaded
2026-03-12 07:47:36,055 [INFO]   Parsing relation2pubtator3.gz...
    relations: 0batch [00:44, ?batch/s]
2026-03-12 07:48:20,658 [INFO]     → 0 relations loaded
2026-03-12 07:48:20,658 [INFO]   Building PMID summary table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03-12 07:49:10,367 [INFO] 
  Index complete:
2026-03-12 07:49:10,370 [INFO]     Total PMIDs indexed:    29,286,980
2026-03-12 07:49:10,383 [INFO]     has_gene:                9,336,406
2026-03-12 07:49:10,395 [INFO]     has_chemical:           21,239,147
2026-03-12 07:49:10,408 [INFO]     has_species:            23,318,240
2026-03-12 07:49:10,416 [INFO]     has_mutation:            1,393,707
2026-03-12 07:49:10,421 [INFO]     has_relation:                    0
2026-03-12 07:49:10,429 [INFO]     has_hv_relation:                 0
2026-03-12 07:49:10,430 [INFO]   DuckDB index saved → pubtator3_index.duckdb


# ResLit: PubTator3 + MeSH Hybrid Classifier

**Four-Signal Filtering Pipeline for AMR Research Papers**

This notebook implements a complete hybrid filtering system combining:
- **PubTator3 NER** — for precise entity extraction (gene, chemical, mutation, species)
- **MeSH Indexing** — for broad curated term coverage
- **Four-signal classifier** — for multi-perspective paper selection

**Assumes PubTator3 files are already downloaded locally.**

## Pipeline Overview

1. **Load & Validate PubTator Files** — Check local gzip files
2. **Build DuckDB Index** — Stream PubTator files into indexed database
3. **Fetch Entrez Metadata** — Get abstracts + MeSH from NCBI
4. **Join PubTator Flags** — Link entity flags into SQLite
5. **Run Classifier** — Apply 4-signal logic
6. **Output Results** — Tier1/2/3 PMID lists + diagnostics


In [1]:
import os
import re
import gzip
import json
import time
import sqlite3
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

import duckdb
import pandas as pd
from tqdm import tqdm
from Bio import Entrez

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

FTP_BASE        = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3"
DOWNLOAD_DIR    = Path("./pubtator3_data")
PUBTATOR_DB     = Path("./pubtator3_index.duckdb")
PAPERS_DB       = Path("./papers.sqlite")
RESULTS_DIR     = Path("./results_test_3000")

PUBTATOR_FILES = {
    "mutation": "mutation2pubtator3.gz",   # 110M
    "gene":     "gene2pubtator3.gz",       # 713M
    "species":  "species2pubtator3.gz",    # 459M
    "chemical": "chemical2pubtator3.gz",   # 1.7G
    "relation": "relation2pubtator3.gz",   # 276M
}

# Relation pairs that strongly suggest a resistance mechanism paper
HIGH_VALUE_RELATION_PAIRS = {
    frozenset(["Gene",    "Chemical"]),
    frozenset(["Variant", "Chemical"]),
    frozenset(["Gene",    "Disease"]),
    frozenset(["Variant", "Disease"]),
}

print("✓ Configuration loaded")
print(f"  PubTator data dir: {DOWNLOAD_DIR}")
print(f"  DuckDB index: {PUBTATOR_DB}")
print(f"  SQLite papers: {PAPERS_DB}")
print(f"  Results output: {RESULTS_DIR}")


✓ Configuration loaded
  PubTator data dir: pubtator3_data
  DuckDB index: pubtator3_index.duckdb
  SQLite papers: papers.sqlite
  Results output: results_test_3000


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFIER TERM SETS
# ─────────────────────────────────────────────────────────────────────────────

# Signal 2 left — AMR mechanism/gene MeSH terms (EXPANDED)
AMR_MECHANISM_MESH = {
    # Beta-lactam resistance
    "beta-Lactamases", "Metallo-beta-Lactamases", "Carbapenemases",
    "Penicillin-Binding Proteins", "Penicillin Resistance",
    # Mobile genetic elements
    "Integrons", "Gene Cassettes", "Plasmids", "Transposons", "Conjugation, Microbial",
    "DNA Transposable Elements", "Bacteriophages", "Prophages",
    # Core bacterial genetics
    "Bacterial Proteins", "Genes, Bacterial", "Multigene Family", "Genes, Essential",
    "Operons", "Regulon", "Gene Dosage",
    # Resistance mechanisms
    "Drug Resistance, Bacterial", "Drug Resistance, Multiple, Bacterial", 
    "Drug Resistance, Microbial", "Microbial Drug Resistance",
    "Cross Resistance", "Methicillin Resistance",
    # Mutations & regulation
    "Mutation", "Mutation, Missense", "Point Mutation", "Frameshift Mutation",
    "Gene Expression Regulation, Bacterial", "Bacterial Transcription",
    # Horizontal gene transfer
    "Genetic Transfer, Horizontal", "Transformation, Bacterial",
    # Efflux & permeability
    "Membrane Transport Proteins", "Bacterial Outer Membrane Proteins",
    "Porins", "ATP-Binding Cassette Transporters", "Multidrug Resistance-Associated Proteins",
    # Biofilm & persistence
    "Biofilms", "Microbial Viability", "Bacterial Persistence",
    # Enzymatic modification
    "Acetyltransferases", "Phosphotransferases", "Methyltransferases",
    # Targets
    "DNA Gyrase", "DNA Topoisomerase IV", "Ribosomal Proteins",
}

# Signal 2 right — drug/resistance MeSH terms (EXPANDED)
DRUG_OR_RESISTANCE_MESH = {
    # Core antibiotic terms
    "Anti-Bacterial Agents", "Anti-Infective Agents", "Antibiotics",
    # Testing & resistance
    "Microbial Sensitivity Tests", "Drug Resistance", "Minimum Inhibitory Concentration",
    "Disk Diffusion Antimicrobial Tests",
    # Beta-lactams
    "Lactams", "beta-Lactams", "Carbapenems", "Penicillins", "Cephalosporins",
    "Monobactams", "beta-Lactamase Inhibitors", "Imipenem", "Meropenem",
    "Ertapenem", "Doripenem", "Ampicillin", "Amoxicillin", "Piperacillin",
    "Ceftriaxone", "Cefotaxime", "Ceftazidime", "Cefepime",
    # Aminoglycosides
    "Aminoglycosides", "Gentamicin", "Amikacin", "Tobramycin", "Streptomycin", "Kanamycin",
    # Fluoroquinolones
    "Fluoroquinolones", "Quinolones", "Ciprofloxacin", "Levofloxacin", "Moxifloxacin",
    # Tetracyclines
    "Tetracyclines", "Tetracycline", "Doxycycline", "Minocycline", "Tigecycline",
    # Macrolides
    "Macrolides", "Erythromycin", "Azithromycin", "Clarithromycin",
    # Glycopeptides
    "Glycopeptides", "Vancomycin", "Teicoplanin",
    # Rifamycins
    "Rifamycins", "Rifampin", "Rifampicin",
    # Polymyxins
    "Polymyxins", "Colistin", "Polymyxin B",
    # Oxazolidinones
    "Oxazolidinones", "Linezolid", "Tedizolid",
    # Folate inhibitors
    "Trimethoprim", "Sulfonamides", "Sulfamethoxazole",
    # Others
    "Chloramphenicol", "Fosfomycin", "Daptomycin", "Nitroimidazoles", "Metronidazole",
    "Clindamycin", "Mupirocin", "Fusidic Acid", "Nitrofurantoin",
}

# Signal 3 — MeSH subheadings indicating genetic characterization (EXPANDED)
GENETICS_SUBHEADINGS = {
    "genetics", "biosynthesis", "isolation & purification", "isolation", 
    "enzymology", "metabolism", "drug effects", "pharmacology",
    "microbiology", "classification", "pathogenicity", "physiology",
}

# Signal 3 — substances that confirm AMR relevance (EXPANDED)
AMR_SUBSTANCES = {
    # Enzymes
    "beta-Lactamases", "Metallo-beta-Lactamases", "Extended-Spectrum beta-Lactamases",
    "carbapenemase", "AmpC beta-Lactamases",
    # Proteins
    "Bacterial Proteins", "Penicillin-Binding Proteins", "Porins",
    "Membrane Proteins", "Efflux Pumps",
    # Nucleic acids
    "DNA, Bacterial", "RNA, Bacterial", "RNA, Ribosomal, 16S", "RNA, Ribosomal, 23S",
    # Mobile elements
    "Plasmids", "Integrons",
    # Drugs
    "Anti-Bacterial Agents", "Anti-Infective Agents", "beta-Lactams",
}


print("✓ Classifier term sets loaded")
print(f"  AMR_MECHANISM_MESH: {len(AMR_MECHANISM_MESH)} terms")
print(f"  DRUG_OR_RESISTANCE_MESH: {len(DRUG_OR_RESISTANCE_MESH)} terms")
print(f"  GENETICS_SUBHEADINGS: {len(GENETICS_SUBHEADINGS)} subheadings")
print(f"  AMR_SUBSTANCES: {len(AMR_SUBSTANCES)} substances")


✓ Classifier term sets loaded
  AMR_MECHANISM_MESH: 48 terms
  DRUG_OR_RESISTANCE_MESH: 69 terms
  GENETICS_SUBHEADINGS: 12 subheadings
  AMR_SUBSTANCES: 19 substances


## 1. Validate Local PubTator Files


In [3]:
print("=" * 70)
print("CHECKING FOR EXISTING PUBTATOR FILES")
print("=" * 70)

# Check what files exist
existing_files = {}
for entity_type, filename in PUBTATOR_FILES.items():
    path = DOWNLOAD_DIR / filename
    if path.exists():
        size_gb = path.stat().st_size / 1e9
        existing_files[entity_type] = (path, size_gb)
        print(f"✓ {filename:<30} {size_gb:>6.2f} GB")
    else:
        print(f"✗ {filename:<30} MISSING")

print()
if len(existing_files) == len(PUBTATOR_FILES):
    print("✓ All PubTator files present! Ready to build index.")
else:
    missing = set(PUBTATOR_FILES.keys()) - set(existing_files.keys())
    print(f"⚠ Missing files: {', '.join(missing)}")
    print("  Download from: https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/")


CHECKING FOR EXISTING PUBTATOR FILES
✓ mutation2pubtator3.gz            0.11 GB
✓ gene2pubtator3.gz                0.75 GB
✓ species2pubtator3.gz             0.48 GB
✓ chemical2pubtator3.gz            1.85 GB
✓ relation2pubtator3.gz            0.29 GB

✓ All PubTator files present! Ready to build index.


## 2. Helper Functions: Stream & Parse PubTator Files


In [4]:
def _stream_entity_file(gz_path: Path, entity_type: str, batch_size: int = 500_000):
    """Stream entity file, yield batches of (pmid, entity_type)."""
    batch = []
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            pmid = parts[0].strip()
            if pmid.isdigit():
                batch.append((pmid, entity_type))
                if len(batch) >= batch_size:
                    yield batch
                    batch = []
    if batch:
        yield batch


def _stream_relation_file(gz_path: Path, batch_size: int = 500_000):
    """Stream relation file, yield batches of (pmid, is_high_value)."""
    batch = []
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 6:
                continue
            pmid    = parts[0].strip()
            e1_type = parts[3].strip()
            e2_type = parts[5].strip()
            if not pmid.isdigit():
                continue
            is_hv = frozenset([e1_type, e2_type]) in HIGH_VALUE_RELATION_PAIRS
            batch.append((pmid, is_hv))
            if len(batch) >= batch_size:
                yield batch
                batch = []
    if batch:
        yield batch

print("✓ Streaming functions defined")


✓ Streaming functions defined


## 3. Build PubTator DuckDB Index


In [5]:
def build_pubtator_index(download_dir: Path = DOWNLOAD_DIR, db_path: Path = PUBTATOR_DB, skip_if_exists: bool = True):
    """Build DuckDB index from PubTator files."""
    
    # ── Skip if index already exists ──
    if skip_if_exists and db_path.exists():
        con = duckdb.connect(str(db_path), read_only=True)
        try:
            total = con.execute("SELECT COUNT(*) FROM pmid_entities").fetchone()[0]
            con.close()
            print(f"\n✓ PubTator index already built at {db_path}")
            print(f"  Contains {total:,} PMIDs — skipping rebuild")
            print(f"  To force rebuild, call with skip_if_exists=False\n")
            return
        except Exception:
            con.close()
            print(f"  Index file exists but is invalid — rebuilding...\n")
    
    print(f"\nBuilding PubTator3 DuckDB index at {db_path}...")
    con = duckdb.connect(str(db_path))

    con.execute("CREATE TABLE IF NOT EXISTS raw_annotations (pmid TEXT, entity_type TEXT)")
    con.execute("CREATE TABLE IF NOT EXISTS raw_relations (pmid TEXT, is_high_value BOOLEAN)")

    # Parse entity files
    for entity_type, filename in [("mutation", PUBTATOR_FILES["mutation"]),
                                   ("gene", PUBTATOR_FILES["gene"]),
                                   ("species", PUBTATOR_FILES["species"]),
                                   ("chemical", PUBTATOR_FILES["chemical"])]:
        gz_path = download_dir / filename
        if not gz_path.exists():
            print(f"  ⚠ Missing: {filename}")
            continue

        print(f"  Parsing {entity_type}...")
        total = 0
        for batch in tqdm(_stream_entity_file(gz_path, entity_type), desc=f"    {entity_type}"):
            df = pd.DataFrame(batch, columns=["pmid", "entity_type"])
            con.execute("INSERT INTO raw_annotations SELECT * FROM df")
            total += len(batch)
        print(f"    → {total:,} annotations")

    # Parse relation file
    relation_path = download_dir / PUBTATOR_FILES["relation"]
    if relation_path.exists():
        print(f"  Parsing relations...")
        total = 0
        for batch in tqdm(_stream_relation_file(relation_path), desc="    relations"):
            df = pd.DataFrame(batch, columns=["pmid", "is_high_value"])
            con.execute("INSERT INTO raw_relations SELECT * FROM df")
            total += len(batch)
        print(f"    → {total:,} relations")

    # Pivot to one row per PMID
    print("  Pivoting to per-PMID format...")
    con.execute("""
        CREATE OR REPLACE TABLE pmid_entities AS
        SELECT
            pmid,
            MAX(CASE WHEN entity_type = 'gene'     THEN TRUE ELSE FALSE END) AS has_gene,
            MAX(CASE WHEN entity_type = 'chemical' THEN TRUE ELSE FALSE END) AS has_chemical,
            MAX(CASE WHEN entity_type = 'species'  THEN TRUE ELSE FALSE END) AS has_species,
            MAX(CASE WHEN entity_type = 'mutation' THEN TRUE ELSE FALSE END) AS has_mutation,
            FALSE AS has_relation,
            FALSE AS has_hv_relation
        FROM raw_annotations
        GROUP BY pmid
    """)

    con.execute("UPDATE pmid_entities SET has_relation = TRUE WHERE pmid IN (SELECT DISTINCT pmid FROM raw_relations)")
    con.execute("UPDATE pmid_entities SET has_hv_relation = TRUE WHERE pmid IN (SELECT DISTINCT pmid FROM raw_relations WHERE is_high_value = TRUE)")

    con.execute("DROP TABLE raw_annotations")
    con.execute("DROP TABLE raw_relations")
    con.execute("CREATE INDEX IF NOT EXISTS idx_pmid ON pmid_entities(pmid)")
    con.execute("ANALYZE")

    # Report
    def count(where=""):
        q = f"SELECT COUNT(*) FROM pmid_entities{(' WHERE '+where) if where else ''}"
        return con.execute(q).fetchone()[0]

    print(f"\n  ✓ PubTator index stats:")
    print(f"    Total PMIDs:          {count():>10,}")
    print(f"    has_gene:             {count('has_gene'):>10,}")
    print(f"    has_chemical:         {count('has_chemical'):>10,}")
    print(f"    has_species:          {count('has_species'):>10,}")
    print(f"    has_mutation:         {count('has_mutation'):>10,}")
    print(f"    has_hv_relation:      {count('has_hv_relation'):>10,}")

    con.close()
    print(f"  ✓ DuckDB index saved → {db_path}\n")

# Build the index (or skip if already exists)
build_pubtator_index()



✓ PubTator index already built at pubtator3_index.duckdb
  Contains 29,286,980 PMIDs — skipping rebuild
  To force rebuild, call with skip_if_exists=False



## 4. Parse PubMed XML & Fetch Entrez Metadata


In [6]:
from xml.etree import ElementTree as ET

def _add_column_if_missing(con, table: str, col_def: str):
    col_name = col_def.split()[0]
    cols = {r[1] for r in con.execute(f"PRAGMA table_info({table})").fetchall()}
    if col_name not in cols:
        con.execute(f"ALTER TABLE {table} ADD COLUMN {col_def}")


def _safe_text(el):
    return el.text.strip() if el is not None and el.text else ""


def _parse_pubmed_xml_rich(xml_handle) -> list[dict]:
    """Parse rich PubMed XML response with broad metadata capture."""
    try:
        tree = ET.parse(xml_handle)
    except ET.ParseError:
        return []

    results = []
    for article in tree.findall(".//PubmedArticle"):
        try:
            pmid = _safe_text(article.find(".//PMID"))
            if not pmid:
                continue

            # Abstract
            abstract_parts = article.findall(".//AbstractText")
            abstract = " ".join(
                (el.get("Label", "") + ": " if el.get("Label") else "") + (el.text or "")
                for el in abstract_parts
            ).strip()

            # MeSH terms with subheadings
            mesh_terms = []
            mesh_subheadings = []
            for mh in article.findall(".//MeshHeading"):
                descriptor = mh.find("DescriptorName")
                if descriptor is None or not descriptor.text:
                    continue
                desc_name = descriptor.text.strip()
                qualifiers = [q.text.strip() for q in mh.findall("QualifierName") if q.text]

                if qualifiers:
                    for q in qualifiers:
                        mesh_terms.append(f"{desc_name} / {q}")
                        mesh_subheadings.append(q.lower())
                else:
                    mesh_terms.append(desc_name)

            # Existing fields
            substances = [n.text.strip() for n in article.findall(".//NameOfSubstance") if n.text]
            pub_types = [pt.text.strip() for pt in article.findall(".//PublicationType") if pt.text]

            pub_year = None
            year_el = article.find(".//PubDate/Year")
            if year_el is not None and year_el.text:
                try:
                    pub_year = int(year_el.text.strip())
                except ValueError:
                    pass
            if pub_year is None:
                medline_el = article.find(".//MedlineDate")
                if medline_el is not None and medline_el.text:
                    match = re.search(r"\b(19|20)\d{2}\b", medline_el.text)
                    if match:
                        pub_year = int(match.group())

            journal = _safe_text(article.find(".//Journal/Title"))

            # Rich metadata fields
            article_title = (article.findtext(".//ArticleTitle", default="") or "").strip()
            language = _safe_text(article.find(".//Language"))
            country = _safe_text(article.find(".//MedlineJournalInfo/Country"))
            journal_iso = _safe_text(article.find(".//ISOAbbreviation"))
            keywords = [k.text.strip() for k in article.findall(".//Keyword") if k.text]

            authors = []
            affiliations = set()
            for a in article.findall(".//Author"):
                last = _safe_text(a.find("LastName"))
                fore = _safe_text(a.find("ForeName"))
                collective = _safe_text(a.find("CollectiveName"))
                name = collective if collective else " ".join([fore, last]).strip()
                affs = [_safe_text(x) for x in a.findall(".//Affiliation") if _safe_text(x)]
                for af in affs:
                    affiliations.add(af)
                if name:
                    authors.append({"name": name, "affiliations": affs})

            article_ids = {}
            for aid in article.findall(".//ArticleId"):
                id_type = aid.attrib.get("IdType", "").strip().lower()
                value = (aid.text or "").strip()
                if id_type and value:
                    article_ids[id_type] = value

            grants = []
            for g in article.findall(".//Grant"):
                grants.append({
                    "grant_id": _safe_text(g.find("GrantID")),
                    "agency": _safe_text(g.find("Agency")),
                    "country": _safe_text(g.find("Country")),
                })

            databanks = []
            for db in article.findall(".//DataBank"):
                db_name = _safe_text(db.find("DataBankName"))
                accessions = [_safe_text(a) for a in db.findall(".//AccessionNumber") if _safe_text(a)]
                databanks.append({"name": db_name, "accessions": accessions})

            references = []
            for ref in article.findall(".//ReferenceList/Reference"):
                citation = _safe_text(ref.find("Citation"))
                ref_ids = [
                    {"type": i.attrib.get("IdType", ""), "value": (i.text or "").strip()}
                    for i in ref.findall(".//ArticleId") if (i.text or "").strip()
                ]
                references.append({"citation": citation, "ids": ref_ids})

            pub_status = _safe_text(article.find(".//PublicationStatus"))
            history = []
            for d in article.findall(".//History/PubMedPubDate"):
                history.append({
                    "status": d.attrib.get("PubStatus", ""),
                    "year": _safe_text(d.find("Year")),
                    "month": _safe_text(d.find("Month")),
                    "day": _safe_text(d.find("Day")),
                })

            comments_corrections = []
            for c in article.findall(".//CommentsCorrections"):
                comments_corrections.append({
                    "ref_type": c.attrib.get("RefType", ""),
                    "source": _safe_text(c.find("RefSource")),
                    "pmid": _safe_text(c.find("PMID")),
                })

            raw_xml = ET.tostring(article, encoding="unicode")

            results.append({
                "pmid": pmid,
                "abstract": abstract,
                "mesh_terms": json.dumps(mesh_terms),
                "mesh_subheadings": json.dumps(mesh_subheadings),
                "substances": json.dumps(substances),
                "pub_types": json.dumps(pub_types),
                "pub_year": pub_year,
                "journal": journal,
                "article_title": article_title,
                "language": language,
                "country": country,
                "journal_iso": journal_iso,
                "keywords": json.dumps(keywords),
                "authors_json": json.dumps(authors),
                "affiliations_json": json.dumps(sorted(affiliations)),
                "article_ids_json": json.dumps(article_ids),
                "doi": article_ids.get("doi", ""),
                "pmc": article_ids.get("pmc", ""),
                "grants_json": json.dumps(grants),
                "databanks_json": json.dumps(databanks),
                "references_json": json.dumps(references),
                "pub_status": pub_status,
                "history_json": json.dumps(history),
                "comments_corrections_json": json.dumps(comments_corrections),
                "raw_xml": raw_xml,
            })
        except Exception:
            continue

    return results


def _fetch_related_pmids_map(pmids: list[str], email: str, api_key=None) -> dict[str, list[str]]:
    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key

    related = {p: [] for p in pmids}
    try:
        handle = Entrez.elink(dbfrom="pubmed", db="pubmed", id=",".join(pmids), cmd="neighbor")
        records = Entrez.read(handle)
        handle.close()

        for rec in records:
            src = str(rec.get("IdList", [""])[0])
            ids = []
            for link_db in rec.get("LinkSetDb", []):
                for link in link_db.get("Link", []):
                    rid = str(link.get("Id", "")).strip()
                    if rid:
                        ids.append(rid)
            related[src] = sorted(set(ids))
    except Exception:
        pass

    return related


def fetch_entrez_metadata_full(pmids: list[str], db_path: Path = PAPERS_DB,
                              email: str = "your@utexas.edu", api_key: str = None,
                              batch_size: int = 200, include_related: bool = True):
    """Fetch rich PubMed metadata and store in SQLite."""
    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key

    con = sqlite3.connect(str(db_path))
    con.execute("""
        CREATE TABLE IF NOT EXISTS papers (
            pmid              TEXT PRIMARY KEY,
            abstract          TEXT,
            mesh_terms        TEXT,
            mesh_subheadings  TEXT,
            substances        TEXT,
            pub_types         TEXT,
            pub_year          INTEGER,
            journal           TEXT,
            has_gene          INTEGER DEFAULT 0,
            has_chemical      INTEGER DEFAULT 0,
            has_species       INTEGER DEFAULT 0,
            has_mutation      INTEGER DEFAULT 0,
            has_hv_relation   INTEGER DEFAULT 0,
            screen_decision   TEXT,
            screen_signal     TEXT
        )
    """)

    for col_def in [
        "article_title TEXT", "language TEXT", "country TEXT", "journal_iso TEXT",
        "keywords TEXT", "authors_json TEXT", "affiliations_json TEXT",
        "article_ids_json TEXT", "doi TEXT", "pmc TEXT",
        "grants_json TEXT", "databanks_json TEXT", "references_json TEXT",
        "pub_status TEXT", "history_json TEXT", "comments_corrections_json TEXT",
        "related_pmids_json TEXT", "raw_xml TEXT",
    ]:
        _add_column_if_missing(con, "papers", col_def)

    con.execute("CREATE INDEX IF NOT EXISTS idx_pmid ON papers(pmid)")
    con.commit()

    existing = {row[0] for row in con.execute("SELECT pmid FROM papers").fetchall()}
    todo = [p for p in pmids if p not in existing]
    print(f"  {len(existing):,} already fetched, {len(todo):,} remaining")

    if not todo:
        print("  ✓ All PMIDs already fetched.")
        con.close()
        return

    delay = 0.11 if api_key else 0.34
    errors = 0

    for i in tqdm(range(0, len(todo), batch_size), desc="  Fetching full PubMed metadata"):
        batch = todo[i:i + batch_size]
        try:
            handle = Entrez.efetch(db="pubmed", id=",".join(batch), rettype="xml", retmode="xml")
            records = _parse_pubmed_xml_rich(handle)
            handle.close()

            related_map = _fetch_related_pmids_map(batch, email=email, api_key=api_key) if include_related else {}
            for rec in records:
                rec["related_pmids_json"] = json.dumps(related_map.get(rec["pmid"], []))

            if records:
                con.executemany("""
                    INSERT OR REPLACE INTO papers (
                        pmid, abstract, mesh_terms, mesh_subheadings, substances, pub_types, pub_year, journal,
                        article_title, language, country, journal_iso, keywords, authors_json, affiliations_json,
                        article_ids_json, doi, pmc, grants_json, databanks_json, references_json, pub_status,
                        history_json, comments_corrections_json, related_pmids_json, raw_xml
                    ) VALUES (
                        :pmid, :abstract, :mesh_terms, :mesh_subheadings, :substances, :pub_types, :pub_year, :journal,
                        :article_title, :language, :country, :journal_iso, :keywords, :authors_json, :affiliations_json,
                        :article_ids_json, :doi, :pmc, :grants_json, :databanks_json, :references_json, :pub_status,
                        :history_json, :comments_corrections_json, :related_pmids_json, :raw_xml
                    )
                """, records)
                con.commit()
        except Exception:
            errors += 1
            if errors > 20:
                print("  Too many Entrez errors — stopping")
                break
            time.sleep(5)
            continue

        time.sleep(delay)

    total = con.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
    print(f"  ✓ Full fetch complete. {total:,} papers in SQLite.\n")
    con.close()


def fetch_entrez_metadata(pmids: list[str], db_path: Path = PAPERS_DB,
                         email: str = "your@utexas.edu", api_key: str = None,
                         batch_size: int = 200):
    """Backward-compatible wrapper."""
    return fetch_entrez_metadata_full(
        pmids=pmids,
        db_path=db_path,
        email=email,
        api_key=api_key,
        batch_size=batch_size,
        include_related=True,
    )


print("✓ Entrez functions defined (rich + related)")

✓ Entrez functions defined (rich + related)


## 5. Join PubTator Flags into SQLite


In [7]:
def join_pubtator_flags(papers_db: Path = PAPERS_DB, pubtator_db: Path = PUBTATOR_DB, batch_size: int = 100_000):
    """Join PubTator entity flags into SQLite papers table."""
    print("Joining PubTator flags into SQLite...")

    sqlite_con = sqlite3.connect(str(papers_db))
    pmids = [row[0] for row in sqlite_con.execute("SELECT pmid FROM papers").fetchall()]

    duck_con = duckdb.connect(str(pubtator_db), read_only=True)

    updated = 0
    for i in tqdm(range(0, len(pmids), batch_size), desc="  Joining"):
        batch = pmids[i:i + batch_size]
        df_batch = pd.DataFrame({"pmid": batch})
        duck_con.register("batch_pmids", df_batch)

        rows = duck_con.execute("""
            SELECT
                b.pmid,
                COALESCE(p.has_gene,        FALSE) AS has_gene,
                COALESCE(p.has_chemical,    FALSE) AS has_chemical,
                COALESCE(p.has_species,     FALSE) AS has_species,
                COALESCE(p.has_mutation,    FALSE) AS has_mutation,
                COALESCE(p.has_hv_relation, FALSE) AS has_hv_relation
            FROM batch_pmids b
            LEFT JOIN pmid_entities p ON b.pmid = p.pmid
        """).fetchall()

        sqlite_con.executemany("""
            UPDATE papers SET
                has_gene=?, has_chemical=?, has_species=?, has_mutation=?, has_hv_relation=?
            WHERE pmid=?
        """, [(r[1], r[2], r[3], r[4], r[5], r[0]) for r in rows])

        updated += len(rows)

    sqlite_con.commit()
    duck_con.close()
    sqlite_con.close()
    print(f"  ✓ Updated {updated:,} rows with PubTator flags\n")

print("✓ join_pubtator_flags defined")


✓ join_pubtator_flags defined


## 6. Four-Signal Classifier


In [8]:
def classify_paper(pmid: str, mesh_terms_json: str, mesh_subheadings_json: str,
                   substances_json: str, pub_types_json: str, has_gene: bool,
                   has_mutation: bool, has_chemical: bool, has_species: bool,
                   has_hv_relation: bool) -> tuple[str, str]:
    """
    Four-signal classifier.
    
    SIGNAL 1: PubTator (gene OR mutation) AND chemical
    SIGNAL 2: MeSH strong: AMR mechanism term AND drug term
    SIGNAL 3: MeSH subheading: genetics subheading + AMR substance
    SIGNAL 4: Species + MeSH: PubTator species AND (AMR + drug MeSH)
    """

    # Parse JSON
    try:
        mesh_terms = json.loads(mesh_terms_json or "[]")
        mesh_subheadings = json.loads(mesh_subheadings_json or "[]")
        substances = json.loads(substances_json or "[]")
        pub_types = json.loads(pub_types_json or "[]")
    except (json.JSONDecodeError, TypeError):
        mesh_terms = mesh_subheadings = substances = pub_types = []

    # Normalize MeSH
    mesh_main = set()
    mesh_subs = set()
    for term in mesh_terms:
        if " / " in term:
            parts = term.split(" / ", 1)
            mesh_main.add(parts[0].strip())
            mesh_subs.add(parts[1].strip().lower())
        else:
            mesh_main.add(term.strip())

    mesh_subs.update(s.lower() for s in mesh_subheadings)
    substances_set = {s.strip() for s in substances}

    # SIGNAL 1: PubTator
    if (has_gene or has_mutation) and has_chemical:
        if has_hv_relation:
            return "pass", "pubtator_hv_relation"
        elif has_mutation:
            return "pass", "pubtator_mutation"
        else:
            return "pass", "pubtator_gene_chemical"

    # SIGNAL 2: MeSH strong
    has_amr_mesh = bool(AMR_MECHANISM_MESH & mesh_main)
    has_drug_mesh = bool(DRUG_OR_RESISTANCE_MESH & mesh_main)
    if has_amr_mesh and has_drug_mesh:
        return "pass", "mesh_strong"

    # SIGNAL 3: MeSH subheading + AMR substance
    has_genetics_sub = bool(GENETICS_SUBHEADINGS & mesh_subs)
    has_amr_substance = bool(AMR_SUBSTANCES & substances_set)
    if has_genetics_sub and (has_amr_substance or has_amr_mesh):
        return "pass", "mesh_genetics_subheading"

    # SIGNAL 4: Species + MeSH
    if has_species and has_amr_mesh and has_drug_mesh:
        return "pass", "species_plus_mesh"

    return "drop", "no_signal"


def run_classifier(papers_db: Path = PAPERS_DB, results_dir: Path = RESULTS_DIR,
                   target_pmids=None, batch_size: int = 50_000):
    """Run classifier and save only pass/not-pass outputs.

    If target_pmids is provided, only those PMIDs are classified.
    """
    results_dir.mkdir(parents=True, exist_ok=True)

    con = sqlite3.connect(str(papers_db))

    target_list = None
    if target_pmids is not None:
        target_list = [str(p).strip() for p in target_pmids if str(p).strip()]
        target_list = list(dict.fromkeys(target_list))  # unique, keep order
        total = len(target_list)
        print(f"  Classifying {total:,} requested PMIDs...\n")
    else:
        total = con.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        print(f"  Classifying {total:,} papers in SQLite...\n")

    passed = []
    not_passed = []
    signal_counter = Counter()
    found_pmids = set()

    if target_list is None:
        # Full-table mode
        offset = 0
        with tqdm(total=total, desc="  Classifying") as pbar:
            while True:
                rows = con.execute(f"""
                    SELECT pmid, mesh_terms, mesh_subheadings, substances, pub_types,
                           has_gene, has_mutation, has_chemical, has_species, has_hv_relation
                    FROM papers LIMIT {batch_size} OFFSET {offset}
                """).fetchall()

                if not rows:
                    break

                updates = []
                for row in rows:
                    pmid, mesh_t, mesh_s, subs, pub_t, h_g, h_m, h_c, h_sp, h_hv = row

                    decision, signal = classify_paper(
                        pmid, mesh_t, mesh_s, subs, pub_t, bool(h_g), bool(h_m),
                        bool(h_c), bool(h_sp), bool(h_hv)
                    )

                    signal_counter[signal] += 1
                    updates.append((decision, signal, pmid))

                    if decision == "pass":
                        passed.append(pmid)
                    else:
                        not_passed.append(pmid)

                con.executemany("UPDATE papers SET screen_decision=?, screen_signal=? WHERE pmid=?", updates)
                con.commit()

                offset += len(rows)
                pbar.update(len(rows))
    else:
        # Requested PMID mode
        # SQLite has a limit of ~999 variables per query, so we need sub-batching
        sqlite_batch_limit = 900
        with tqdm(total=total, desc="  Classifying") as pbar:
            for i in range(0, len(target_list), sqlite_batch_limit):
                batch_pmids = target_list[i:i + sqlite_batch_limit]
                if not batch_pmids:
                    continue

                placeholders = ",".join(["?"] * len(batch_pmids))
                rows = con.execute(
                    f"""
                    SELECT pmid, mesh_terms, mesh_subheadings, substances, pub_types,
                           has_gene, has_mutation, has_chemical, has_species, has_hv_relation
                    FROM papers
                    WHERE pmid IN ({placeholders})
                    """,
                    tuple(batch_pmids),
                ).fetchall()

                updates = []
                for row in rows:
                    pmid, mesh_t, mesh_s, subs, pub_t, h_g, h_m, h_c, h_sp, h_hv = row
                    found_pmids.add(pmid)

                    decision, signal = classify_paper(
                        pmid, mesh_t, mesh_s, subs, pub_t, bool(h_g), bool(h_m),
                        bool(h_c), bool(h_sp), bool(h_hv)
                    )

                    signal_counter[signal] += 1
                    updates.append((decision, signal, pmid))

                    if decision == "pass":
                        passed.append(pmid)
                    else:
                        not_passed.append(pmid)

                if updates:
                    con.executemany("UPDATE papers SET screen_decision=?, screen_signal=? WHERE pmid=?", updates)
                    con.commit()

                pbar.update(len(batch_pmids))

        missing_pmids = [p for p in target_list if p not in found_pmids]
        if missing_pmids:
            print(f"  ⚠ {len(missing_pmids):,} requested PMIDs were not found in SQLite; marking as not_passed")
            not_passed.extend(missing_pmids)

    con.close()

    # Save only two files
    def save(pmids, name):
        path = results_dir / f"{name}.txt"
        seen = set()
        unique = [p for p in pmids if not (p in seen or seen.add(p))]
        with open(path, "w") as f:
            f.write("\n".join(unique))
        print(f"    {len(unique):>8,}  →  {path.name}")
        return unique

    print(f"\n{'='*65}")
    print("  CLASSIFICATION RESULTS")
    print(f"{'='*65}")
    print(f"  Requested papers:      {total:>8,}")
    print(f"{'='*65}")
    print("  OUTPUT FILES:")

    passed_unique = save(passed, "passed")
    not_passed_unique = save(not_passed, "not_passed")

    print(f"{'='*65}")
    print("  SIGNAL BREAKDOWN:")
    for signal, count in signal_counter.most_common():
        print(f"    {signal:<35} {count:>8,}")
    print(f"{'='*65}\n")

    return {
        "passed": passed_unique,
        "not_passed": not_passed_unique,
        "signal_counter": signal_counter,
    }

print("✓ Classifier functions defined")

✓ Classifier functions defined


## TEST CONFIGURATION: Run on 3000 Known Papers

Enter your configuration below:


In [9]:
print("="*70)
print("ResLit - Hybrid PubTator + MeSH Filter")
print("="*70)
print("\nTesting on ~ 2M AMR papers")
print("Using existing PubTator3 index (no re-download)")
print("Year range: 1960 - present (no filtering)\n")

# Configuration for testing
CONFIG = {
    "seed_pmids": "/work/11252/skulakis/projects/reslit/amr_pmids_genetics2.txt",  # Path to file with PMIDs to classify (one PMID per line)
    "ncbi_api_key": "23948dc88d92c5002ac5cb8a91ac19cdc509",                      # Optional: set your key to speed up
    "download_dir": Path("./pubtator3_data"),
    "pubtator_db": Path("./pubtator3_index.duckdb"),
    "papers_db": Path("./papers.sqlite"),
    "results_dir": Path("./results_amr_pmids_genetics"),
}

print("Configuration:")
print(f"  Seed PMIDs:      {CONFIG['seed_pmids']}")
print(f"  Skip download:   TRUE (using existing PubTator data)")
print(f"  Skip index:      TRUE (using existing DuckDB index)")
print(f"  Fetch Entrez:    YES (will fetch/update papers.sqlite)")
print(f"  Join flags:      YES (will add PubTator flags)")
print(f"  Results dir:     {CONFIG['results_dir']}")
print(f"\nEstimated time:")
print(f"  Fetch Entrez:    ~10-30 min (3000 papers @ 2-3 req/sec)")
print(f"  Classify:        ~1-2 min")
print(f"  Total:           ~20-40 minutes WITHOUT API key")
if CONFIG['ncbi_api_key']:
    print(f"  With API key:    ~5-10 minutes (10 req/sec)")
else:
    print(f"  [Optional speed-up: add your NCBI API key above]")
    print(f"   Get it at: https://www.ncbi.nlm.nih.gov/account/")
print("="*70 + "\n")

ResLit - Hybrid PubTator + MeSH Filter

Testing on ~ 2M AMR papers
Using existing PubTator3 index (no re-download)
Year range: 1960 - present (no filtering)

Configuration:
  Seed PMIDs:      /work/11252/skulakis/projects/reslit/amr_pmids_genetics2.txt
  Skip download:   TRUE (using existing PubTator data)
  Skip index:      TRUE (using existing DuckDB index)
  Fetch Entrez:    YES (will fetch/update papers.sqlite)
  Join flags:      YES (will add PubTator flags)
  Results dir:     results_amr_pmids_genetics

Estimated time:
  Fetch Entrez:    ~10-30 min (3000 papers @ 2-3 req/sec)
  Classify:        ~1-2 min
  Total:           ~20-40 minutes WITHOUT API key
  With API key:    ~5-10 minutes (10 req/sec)



## EXECUTE: Run the Pipeline


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# EXECUTE PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def _now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _fmt_elapsed(seconds):
    seconds = int(round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

pipeline_start = time.perf_counter()
pipeline_start_wall = _now_str()

print(f"Pipeline started at: {pipeline_start_wall}")
print("Loading seed PMIDs...")
with open(CONFIG["seed_pmids"]) as f:
    seed_pmids = [line.strip() for line in f if line.strip()]
print(f"✓ Loaded {len(seed_pmids):,} seed PMIDs\n")

# Step 1: Fetch rich Entrez metadata
print("=" * 70)
print("STEP 1: Fetching full Entrez metadata (+ related PMIDs)")
print(f"Started: {_now_str()}")
print("=" * 70)
step_start = time.perf_counter()
fetch_entrez_metadata_full(
    pmids=seed_pmids,
    db_path=CONFIG["papers_db"],
    api_key=CONFIG["ncbi_api_key"],
    include_related=True
)
print(f"Finished: {_now_str()} | Elapsed: {_fmt_elapsed(time.perf_counter() - step_start)}\n")

# Step 2: Join PubTator flags
print("=" * 70)
print("STEP 2: Joining PubTator entity flags")
print(f"Started: {_now_str()}")
print("=" * 70)
step_start = time.perf_counter()
join_pubtator_flags(
    papers_db=CONFIG["papers_db"],
    pubtator_db=CONFIG["pubtator_db"]
)
print(f"Finished: {_now_str()} | Elapsed: {_fmt_elapsed(time.perf_counter() - step_start)}\n")

# Step 3: Run classifier (seed list only)
print("=" * 70)
print("STEP 3: Running classifier (seed PMIDs only)")
print(f"Started: {_now_str()}")
print("=" * 70)
step_start = time.perf_counter()
results = run_classifier(
    papers_db=CONFIG["papers_db"],
    results_dir=CONFIG["results_dir"],
    target_pmids=seed_pmids
)
print(f"Finished: {_now_str()} | Elapsed: {_fmt_elapsed(time.perf_counter() - step_start)}\n")

pipeline_elapsed = time.perf_counter() - pipeline_start
pipeline_end_wall = _now_str()

print("\n" + "=" * 70)
print("✓ PIPELINE COMPLETE")
print("=" * 70)
print(f"Started at:       {pipeline_start_wall}")
print(f"Finished at:      {pipeline_end_wall}")
print(f"Total elapsed:    {_fmt_elapsed(pipeline_elapsed)}")
print(f"\nResults saved to: {CONFIG['results_dir']}/")
print("\nOutput files:")
print("  passed.txt")
print("  not_passed.txt")
print("=" * 70)

Pipeline started at: 2026-03-12 10:48:54
Loading seed PMIDs...


✓ Loaded 2,057,493 seed PMIDs

STEP 1: Fetching full Entrez metadata (+ related PMIDs)
Started: 2026-03-12 10:48:54
  651,241 already fetched, 1,406,446 remaining


  Fetching full PubMed metadata: 100%|██████████| 7033/7033 [4:08:27<00:00,  2.12s/it]   


  ✓ Full fetch complete. 2,055,617 papers in SQLite.

Finished: 2026-03-12 14:58:38 | Elapsed: 04:09:44

STEP 2: Joining PubTator entity flags
Started: 2026-03-12 14:58:38
Joining PubTator flags into SQLite...


  Joining: 100%|██████████| 21/21 [1:34:41<00:00, 270.54s/it]


  ✓ Updated 2,055,617 rows with PubTator flags

Finished: 2026-03-12 16:33:21 | Elapsed: 01:34:44

STEP 3: Running classifier (seed PMIDs only)
Started: 2026-03-12 16:33:21
  Classifying 2,057,493 requested PMIDs...



  Classifying: 100%|██████████| 2057493/2057493 [16:34<00:00, 2069.77it/s]


  ⚠ 2,070 requested PMIDs were not found in SQLite; marking as not_passed

  CLASSIFICATION RESULTS
  Requested papers:      2,057,493
  OUTPUT FILES:
    1,543,369  →  passed.txt
     514,124  →  not_passed.txt
  SIGNAL BREAKDOWN:
    mesh_genetics_subheading             625,909
    pubtator_gene_chemical               533,115
    no_signal                            512,054
    pubtator_mutation                    250,565
    mesh_strong                          133,780

Finished: 2026-03-12 16:49:57 | Elapsed: 00:16:36


✓ PIPELINE COMPLETE
Started at:       2026-03-12 10:48:54
Finished at:      2026-03-12 16:49:57
Total elapsed:    06:01:04

Results saved to: results_amr_pmids_genetics/

Output files:
  passed.txt
  not_passed.txt


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK: Seed PMIDs that did NOT pass
# ══════════════════════════════════════════════════════════════════════════════

seed_file = Path(CONFIG["seed_pmids"])
with open(seed_file) as f:
    seed_pmids_all = [line.strip() for line in f if line.strip()]
seed_set = set(seed_pmids_all)

if not seed_set:
    print("Seed list is empty.")
else:
    con = sqlite3.connect(str(CONFIG["papers_db"]))

    # Batch query to avoid SQLite variable limit (~999)
    rows = []
    seed_list = list(seed_set)
    sqlite_batch_limit = 900
    
    for i in range(0, len(seed_list), sqlite_batch_limit):
        batch = seed_list[i:i + sqlite_batch_limit]
        placeholders = ",".join(["?"] * len(batch))
        batch_rows = con.execute(
            f"""
            SELECT pmid, COALESCE(screen_decision, 'not_scored') AS screen_decision,
                   COALESCE(screen_signal, '') AS screen_signal
            FROM papers
            WHERE pmid IN ({placeholders})
              AND COALESCE(screen_decision, 'drop') != 'pass'
            """,
            tuple(batch)
        ).fetchall()
        rows.extend(batch_rows)
    
    con.close()

    not_passed_pmids = [r[0] for r in rows]

    print("=" * 80)
    print("SEED PMID PASS/FAIL CHECK")
    print("=" * 80)
    print(f"Seed PMIDs total:      {len(seed_set):,}")
    print(f"Did NOT pass:          {len(not_passed_pmids):,}")
    print(f"Passed:                {len(seed_set) - len(not_passed_pmids):,}")
    print("=" * 80)

    # Save full details + PMID list
    details_file = CONFIG["results_dir"] / "seed_pmids_not_passed_details.tsv"
    pmids_file = CONFIG["results_dir"] / "seed_pmids_not_passed.txt"

    CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)

    with open(details_file, "w") as f:
        f.write("pmid\tscreen_decision\tscreen_signal\n")
        for pmid, decision, signal in rows:
            f.write(f"{pmid}\t{decision}\t{signal}\n")

    with open(pmids_file, "w") as f:
        f.write("\n".join(not_passed_pmids))

    print(f"Saved details: {details_file}")
    print(f"Saved PMID list: {pmids_file}")

    if rows:
        print("\nFirst 20 not-passed PMIDs:")
        for pmid, decision, signal in rows[:20]:
            print(f"  {pmid}  |  {decision}  |  {signal}")
    else:
        print("\nAll seed PMIDs passed.")

KeyboardInterrupt: 

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# POST-CLASSIFIER CLEANUP: Remove wrong organism + non-English
# ══════════════════════════════════════════════════════════════════════════════

# ── MeSH terms that indicate NON-bacterial organisms ──
NON_BACTERIAL_ORGANISM_MESH = {
    # Fungi
    "Fungi", "Candida", "Candida albicans", "Candida glabrata", "Candida auris",
    "Aspergillus", "Aspergillus fumigatus", "Cryptococcus", "Cryptococcus neoformans",
    "Saccharomyces", "Pneumocystis", "Fusarium", "Trichophyton", "Dermatophytes",
    "Antifungal Agents", "Drug Resistance, Fungal",
    # Viruses
    "Viruses", "HIV", "HIV-1", "HIV-2", "Influenza A virus", "Influenza B virus",
    "Hepatitis B virus", "Hepatitis C virus", "Herpesvirus", "Cytomegalovirus",
    "SARS-CoV-2", "Coronavirus", "Antiviral Agents", "Drug Resistance, Viral",
    # Parasites
    "Plasmodium", "Plasmodium falciparum", "Plasmodium vivax",
    "Trypanosoma", "Leishmania", "Toxoplasma", "Giardia", "Cryptosporidium",
    "Helminths", "Malaria", "Antiparasitic Agents", "Drug Resistance, Parasites",
}

# ── MeSH terms that CONFIRM bacterial context (safe override) ──
BACTERIAL_CONFIRM_MESH = {
    "Bacteria", "Gram-Positive Bacteria", "Gram-Negative Bacteria",
    "Staphylococcus", "Staphylococcus aureus", "Streptococcus", "Enterococcus",
    "Escherichia coli", "Klebsiella", "Klebsiella pneumoniae",
    "Pseudomonas", "Pseudomonas aeruginosa", "Acinetobacter", "Acinetobacter baumannii",
    "Salmonella", "Mycobacterium", "Mycobacterium tuberculosis",
    "Haemophilus", "Neisseria", "Clostridioides difficile", "Clostridium",
    "Enterobacteriaceae", "Bacteroidetes", "Listeria", "Campylobacter",
    "Helicobacter pylori", "Burkholderia", "Stenotrophomonas",
    "Drug Resistance, Bacterial", "Drug Resistance, Multiple, Bacterial",
    "Anti-Bacterial Agents", "beta-Lactamases", "Bacterial Proteins",
    "Genes, Bacterial", "Gene Expression Regulation, Bacterial",
}

def post_classifier_cleanup(
    papers_db: Path = PAPERS_DB,
    results_dir: Path = RESULTS_DIR,
    passed_file: str = "passed.txt",
    remove_non_english: bool = True,
    remove_wrong_organism: bool = True,
):
    results_dir.mkdir(parents=True, exist_ok=True)

    # Load current passed list
    passed_path = results_dir / passed_file
    with open(passed_path) as f:
        passed_pmids = [line.strip() for line in f if line.strip()]

    print("=" * 70)
    print("POST-CLASSIFIER CLEANUP")
    print("=" * 70)
    print(f"  Input passed PMIDs:     {len(passed_pmids):>8,}")

    con = sqlite3.connect(str(papers_db))
    
    # Batch query to avoid SQLite variable limit (~999)
    rows = []
    sqlite_batch_limit = 900
    
    for i in range(0, len(passed_pmids), sqlite_batch_limit):
        batch = passed_pmids[i:i + sqlite_batch_limit]
        placeholders = ",".join(["?"] * len(batch))
        batch_rows = con.execute(
            f"""
            SELECT pmid, language, mesh_terms
            FROM papers
            WHERE pmid IN ({placeholders})
            """,
            tuple(batch),
        ).fetchall()
        rows.extend(batch_rows)
    
    con.close()

    # Build lookup
    meta = {r[0]: {"language": r[1] or "", "mesh_terms": r[2] or "[]"} for r in rows}

    removed_language = []
    removed_organism = []
    kept = []

    for pmid in passed_pmids:
        info = meta.get(pmid)
        if info is None:
            kept.append(pmid)
            continue

        # ── Language filter ──
        if remove_non_english and info["language"] and info["language"].lower() != "eng":
            removed_language.append(pmid)
            continue

        # ── Organism filter ──
        if remove_wrong_organism:
            try:
                mesh_list = json.loads(info["mesh_terms"])
            except Exception:
                mesh_list = []

            mesh_main = set()
            for term in mesh_list:
                if " / " in term:
                    mesh_main.add(term.split(" / ", 1)[0].strip())
                else:
                    mesh_main.add(term.strip())

            has_bacterial = bool(BACTERIAL_CONFIRM_MESH & mesh_main)
            has_non_bacterial = bool(NON_BACTERIAL_ORGANISM_MESH & mesh_main)

            # Remove only if non-bacterial AND no bacterial confirmation
            if has_non_bacterial and not has_bacterial:
                removed_organism.append(pmid)
                continue

        kept.append(pmid)

    # Save cleaned passed list
    cleaned_path = results_dir / "passed_cleaned.txt"
    with open(cleaned_path, "w") as f:
        f.write("\n".join(kept))

    # Save removal lists for audit
    if removed_language:
        lang_path = results_dir / "removed_non_english.txt"
        with open(lang_path, "w") as f:
            f.write("\n".join(removed_language))

    if removed_organism:
        org_path = results_dir / "removed_wrong_organism.txt"
        with open(org_path, "w") as f:
            f.write("\n".join(removed_organism))

    print(f"  Removed non-English:    {len(removed_language):>8,}  → removed_non_english.txt")
    print(f"  Removed wrong organism: {len(removed_organism):>8,}  → removed_wrong_organism.txt")
    print(f"  Final kept:             {len(kept):>8,}  → passed_cleaned.txt")
    print("=" * 70)

    return {
        "kept": kept,
        "removed_language": removed_language,
        "removed_organism": removed_organism,
    }

# ── Run cleanup ──
cleanup_results = post_classifier_cleanup(
    papers_db=CONFIG["papers_db"],
    results_dir=CONFIG["results_dir"],
    remove_non_english=True,
    remove_wrong_organism=True,
)

POST-CLASSIFIER CLEANUP
  Input passed PMIDs:     1,543,369
  Removed non-English:      79,380  → removed_non_english.txt
  Removed wrong organism:   43,402  → removed_wrong_organism.txt
  Final kept:             1,420,587  → passed_cleaned.txt
